# Chirality Atlas: Star Ascidian Edition

**UCSD Vibe Coding Active Matter 2026 Hackathon**

Can local physical rules produce a living star pattern?

This notebook walks through the full modeling pipeline for *Botryllus schlosseri*,
the star ascidian -- a colonial tunicate that organizes its zooids into radial star-shaped systems.
We build a two-layer computational model from scratch, measure how well it reproduces
the biological target, and explore what happens when we tune chirality.

**Sections:**
1. Setup
2. Target pattern: star ascidian colony
3. Observe: visual features of the target
4. Hypothesize: two-layer mechanism
5. Simulate baseline models
6. Build the hybrid model
7. Measure: quantitative metrics
8. Explore: phase diagrams
9. Compare: target vs simulation
10. Explain: what the model explains and what it does not
11. LLM workflow: prompt-run-test-modify-critique
12. 5-slide checklist

**Estimated runtime:** 5-15 minutes depending on hardware (defaults use small grids).

---
## 1. Setup

In [ ]:
# Install dependencies (run this cell first in Colab)
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + list(packages))

try:
    import scipy
except ImportError:
    pip_install("scipy")

try:
    import PIL
except ImportError:
    pip_install("Pillow")

print("Dependencies ready.")

In [ ]:
# Clone repo if running in Colab
import os

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(os.environ.get("JUPYTER_SERVER_ROOT", ""))

if IN_COLAB:
    if not os.path.exists("chirality"):
        subprocess.check_call(["git", "clone", "https://github.com/agentjakey/chirality.git"])
    os.chdir("chirality")
    sys.path.insert(0, "src")
    print("Running in Colab -- repo cloned.")
else:
    # Running locally -- add src to path
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    src_path = os.path.join(repo_root, "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    print(f"Running locally. src path: {src_path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Ellipse

# Project imports
from chirality.star_ascidian.hybrid_model import simulate_star_ascidian_colony, PRESETS
from chirality.star_ascidian.center_field import generate_star_centers
from chirality.star_ascidian.zooid_agents import simulate_zooid_agents
from chirality.star_ascidian import metrics as star_metrics
from chirality.model_library.gierer_meinhardt import simulate_gierer_meinhardt
from chirality.model_library.active_particles import simulate_abp

# Visualization constants
BG = "#F7F3EA"
ACCENT = "#C15A3A"
GREEN = "#315C4C"
INK = "#1F2421"
BORDER = "#DDD5C8"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": "#FFFFFF",
    "font.family": "serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 42
print("Imports OK. PRESETS available:", list(PRESETS.keys()))

---
## 2. Target Pattern: Star Ascidian Colony

*Botryllus schlosseri* is a colonial tunicate found in shallow marine waters.
Individual animals (zooids) arrange themselves into star-shaped systems:
each system has a shared central atrium and approximately 5-10 zooids
radiating outward in discrete arms. Multiple systems tile the colony with
characteristic spacing.

**Target feature checklist:**

| Feature | Biological observation |
|---|---|
| Multiple star centers | 3-10 systems per colony tile |
| Radial arm structure | ~7 arms per star, extending from a central atrium |
| Regular center spacing | Stars do not merge; characteristic distance maintained |
| Approximate arm uniformity | Arms distributed roughly equally around the center |
| Zooid confinement | Each zooid belongs to one star; limited cross-star migration |
| Chirality | Some colonies show consistent arm rotation direction |

We will measure each of these features quantitatively at the end of the simulation.

In [ ]:
def draw_star_ascidian(ax, cx, cy, r_inner=0.12, r_outer=0.35, n_arms=7,
                       arm_color=ACCENT, center_color=GREEN, alpha=0.85):
    """Draw a single schematic star ascidian on ax."""
    ax.add_patch(plt.Circle((cx, cy), r_inner, color=center_color, alpha=alpha, zorder=3))
    for k in range(n_arms):
        angle = 2 * np.pi * k / n_arms
        arm_cx = cx + (r_inner + r_outer) * 0.5 * np.cos(angle)
        arm_cy = cy + (r_inner + r_outer) * 0.5 * np.sin(angle)
        e = Ellipse(
            (arm_cx, arm_cy),
            width=r_outer * 0.45,
            height=r_outer * 1.0,
            angle=np.degrees(angle),
            color=arm_color, alpha=alpha * 0.8, zorder=2,
        )
        ax.add_patch(e)

fig, ax = plt.subplots(figsize=(5, 5), facecolor=BG)
ax.set_facecolor(BG)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Target: Botryllus schlosseri star ascidian\n(schematic, not to scale)",
             color=INK, fontsize=11)

centers_schematic = [(0.28, 0.28), (0.72, 0.28), (0.28, 0.72), (0.72, 0.72)]
for (cx, cy) in centers_schematic:
    draw_star_ascidian(ax, cx, cy)

plt.tight_layout()
plt.show()
print("Target schematic: 4 star systems, each with 7 radial arms.")

---
## 3. Observe: Visual Features of the Target

Before building any model, write down what you actually see.
This forces you to define measurable targets before fitting.

**From the schematic and literature images of Botryllus:**

- There are multiple star systems arranged across the colony surface.
- Each star has a bright, compact central atrium (shared opening for filtration).
- Arms radiate outward from that center at roughly equal angular intervals.
- Stars maintain a consistent spacing -- they do not overlap or merge.
- Individual zooids are small relative to arm length; arms look solid in photographs.
- Some species/strains show consistent left or right twist in arm orientation.

**What this tells us about the mechanism:**

- The pattern is *periodic* in space, suggesting a self-organization process
  with a characteristic length scale (typical of reaction-diffusion or inhibition-mediated spacing).
- Arms imply *radial symmetry breaking* at each center: some local force must push zooids
  apart angularly, not just radially.
- Confinement of zooids to their home star suggests an *attraction* to their center
  that prevents drift across the colony.
- Chirality, when present, is *collective*: all arms of a given star rotate the same direction.

---
## 4. Hypothesize: Two-Layer Mechanism

We propose a two-layer model that separates center placement from arm formation.

**Layer 1: Activator-Inhibitor Field (Turing mechanism)**

A Gierer-Meinhardt reaction-diffusion system produces spatially periodic spots.
These spots set the star center positions.

Why GM? The short-range activation / long-range inhibition structure naturally produces
a quasi-regular array of spots with characteristic spacing set by the ratio Dh/Da.
This mimics the spacing regularity of Botryllus stars without requiring
explicit distance rules between zooids.

    da/dt = Da * lap(a) + rho * a^2 / (h * (1 + kappa * a^2)) - mu_a * a + rho_0
    dh/dt = Dh * lap(h) + rho * a^2 - mu_h * h

**Layer 2: Active Zooid Agents (arm formation)**

Each zooid is an active self-propelled particle. Forces:

    dx/dt = v0 * cos(theta) + F_attract + F_radial + F_angular + F_ev
    dy/dt = v0 * sin(theta) + ...same
    dtheta/dt = omega + sqrt(2*Dr) * xi(t)

- **Center attraction** (F_attract): soft pull toward assigned center. Prevents cross-star drift.
- **Radial spring** (F_radial): pushes zooids to a target ring at radius r_target. Creates the arm lobes.
- **Angular repulsion** (F_angular): pushes zooids from different arms apart in the tangential direction. Creates even arm spacing.
- **Excluded volume** (F_ev): soft repulsion between zooids at close range.
- **Chirality** (omega): net rotation rate of orientation. A nonzero omega twists the arm pattern.

**Hypothesis:** these five forces together are sufficient to reproduce the geometry of Botryllus stars.

---
## 5. Simulate Baseline Models

Before running the full hybrid model, we verify the two layers separately.

### 5a. Gierer-Meinhardt spots

In [ ]:
# Fast GM run for demonstration (small grid)
gm_result = simulate_gierer_meinhardt(
    N=32, L=10.0,
    Da=0.05, Dh=5.0, mu_a=0.05, mu_h=0.05,
    rho=0.1, rho_0=0.001, kappa=0.1,
    dt=0.5, n_steps=1500, n_snapshots=4,
    seed=SEED,
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), facecolor=BG)

for ax in axes:
    ax.set_facecolor("#FFFFFF")

axes[0].imshow(gm_result.snapshots_u[0], cmap="YlOrRd", origin="lower", aspect="equal")
axes[0].set_title("t=0 (initial noise)", color=INK)

mid = len(gm_result.snapshots_u) // 2
axes[1].imshow(gm_result.snapshots_u[mid], cmap="YlOrRd", origin="lower", aspect="equal")
axes[1].set_title("t=mid (pattern forming)", color=INK)

axes[2].imshow(gm_result.u_final, cmap="YlOrRd", origin="lower", aspect="equal")
axes[2].set_title("t=final (Turing spots)", color=INK)

for ax in axes:
    ax.axis("off")

fig.suptitle("Layer 1: Gierer-Meinhardt activator field", color=INK, y=1.01)
plt.tight_layout()
plt.show()

print(f"GM run complete. Pattern strength: {gm_result.pattern_strength:.3f}")
print(f"Number of spots (clusters): {gm_result.cluster_count}")
print("These spots will become star center positions.")

### 5b. Active particles baseline

In [ ]:
# Basic active Brownian particles (no chirality, no forces)
abp_result = simulate_abp(
    N=80, L=10.0,
    v0=0.5, Dr=0.3,
    dt=0.05, n_steps=500, n_snapshots=4,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), facecolor=BG)
for ax in axes:
    ax.set_facecolor("#FFFFFF")

pos_init = abp_result.positions[0]
axes[0].scatter(pos_init[:, 0], pos_init[:, 1], c=ACCENT, s=18, alpha=0.7)
axes[0].set_title("ABP: t=0", color=INK)
axes[0].set_xlim(0, 10)
axes[0].set_ylim(0, 10)
axes[0].set_aspect("equal")

pos_final = abp_result.positions[-1]
axes[1].scatter(pos_final[:, 0], pos_final[:, 1], c=GREEN, s=18, alpha=0.7)
axes[1].set_title("ABP: t=final (diffuse gas)", color=INK)
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 10)
axes[1].set_aspect("equal")

plt.suptitle("Layer 2 basis: Active Brownian Particles\n(no forces = uniform gas)",
             color=INK, y=1.02)
plt.tight_layout()
plt.show()

print("Without forces, active particles spread uniformly -- no arms form.")
print("This is why we need the additional forces in our zooid model.")

---
## 6. Build the Hybrid Model

Now we run both layers together using the `clean_star_systems` preset.

**What happens:**
1. GM field runs for 3000 steps. Spots form and stabilize.
2. Local maxima are extracted as star center positions.
3. Zooid agents are initialized around those centers in discrete arm groups.
4. Agent dynamics run for 400 steps. Arms form and tighten.

This will take 30-90 seconds depending on hardware.

In [ ]:
print("Running clean_star_systems preset...")
print("(GM: 3000 steps at N=64; Agents: 400 steps)")

result_clean = simulate_star_ascidian_colony(
    preset="clean_star_systems",
    seed=SEED,
    n_snapshots=10,
)

print(f"Done. Found {result_clean.zooid.K} star centers.")
print(f"Total zooid agents: {result_clean.zooid.N}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), facecolor=BG)
for ax in axes:
    ax.set_facecolor("#FFFFFF")

# Panel 1: GM activator field
axes[0].imshow(result_clean.field, cmap="YlOrRd", origin="lower", aspect="equal")
axes[0].set_title("Layer 1: GM activator field", color=INK)
axes[0].axis("off")

# Panel 2: Center positions on field
L = result_clean.params["L"]
N = result_clean.params["grid_size"]
axes[1].imshow(result_clean.field, cmap="YlOrRd", origin="lower", aspect="equal",
               extent=[0, L, 0, L], alpha=0.4)
cxs = result_clean.centers[:, 0]
cys = result_clean.centers[:, 1]
axes[1].scatter(cxs, cys, c=GREEN, s=80, zorder=5, edgecolors="white", linewidths=1.2)
axes[1].set_title(f"Star center positions (K={result_clean.zooid.K})", color=INK)
axes[1].set_xlim(0, L)
axes[1].set_ylim(0, L)
axes[1].set_aspect("equal")

# Panel 3: Final agent positions colored by center
pos = result_clean.zooid.positions[-1]
assignments = result_clean.zooid.assignments
K = result_clean.zooid.K
cmap_agents = plt.cm.get_cmap("tab10", K)
for k in range(K):
    mask = assignments == k
    axes[2].scatter(pos[mask, 0], pos[mask, 1], c=[cmap_agents(k)], s=14, alpha=0.75)
axes[2].scatter(cxs, cys, c="white", s=60, marker="*", zorder=5, edgecolors=INK, linewidths=0.8)
axes[2].set_title("Layer 2: Zooid agents (colored by star)", color=INK)
axes[2].set_xlim(0, L)
axes[2].set_ylim(0, L)
axes[2].set_aspect("equal")

fig.suptitle("Hybrid model: clean_star_systems preset", color=INK, fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Measure: Quantitative Metrics

We use five metrics to score how star-like the simulation looks.

| Metric | What it measures | Range |
|---|---|---|
| radial_order | Fraction of agents at target radius | [0, 1] |
| arm_count | Mean detected arms per center | integer |
| angular_uniformity | Even arm spacing (1 - CV of inter-arm angles) | [0, 1] |
| star_likeness | Composite of the above three | [0, 1] |
| swirl_score | Net tangential velocity around centers (chirality proxy) | [-1, 1] |
| fragmentation | Fraction of agents far from their center | [0, 1] |
| merge_score | Cross-center proximity (stars merging) | [0, 1] |

In [ ]:
m = result_clean.metrics

print("=== Metrics: clean_star_systems ===")
print(f"  star_likeness_score:  {m['star_likeness_score']:.3f}   (target: 1.0)")
print(f"  radial_order:         {m['radial_order']:.3f}   (target: 1.0)")
print(f"  arm_count_mean:       {m['arm_count_mean']:.1f}    (target: 7.0)")
print(f"  angular_uniformity:   {m['angular_uniformity']:.3f}   (target: 1.0)")
print(f"  swirl_score:          {m['swirl_score']:.3f}   (target: ~0 for no chirality)")
print(f"  fragmentation:        {m['fragmentation']:.3f}   (target: 0.0)")
print(f"  merge_score:          {m['merge_score']:.3f}   (target: 0.0)")
print()
print("Matches:")
for s in m["matches"]:
    print(f"  [OK] {s}")
print()
print("Failures:")
for s in m["failures"]:
    print(f"  [--] {s}")
if not m["failures"]:
    print("  (none)")

In [ ]:
metric_names = [
    "star_likeness", "radial_order", "angular_uniformity",
    "1-fragmentation", "1-merge_score"
]
metric_values = [
    m["star_likeness_score"],
    m["radial_order"],
    m["angular_uniformity"],
    1.0 - m["fragmentation"],
    1.0 - m["merge_score"],
]
target_values = [1.0] * len(metric_names)

fig, ax = plt.subplots(figsize=(8, 3.5), facecolor=BG)
ax.set_facecolor("#FFFFFF")
x = np.arange(len(metric_names))
w = 0.35
ax.bar(x - w/2, target_values, w, color=BORDER, label="Target", zorder=2)
ax.bar(x + w/2, metric_values, w, color=GREEN, label="Simulation", zorder=2)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=20, ha="right", color=INK)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score", color=INK)
ax.set_title("Target vs simulation metrics: clean_star_systems", color=INK)
ax.legend()
ax.axhline(1.0, color=ACCENT, lw=1, ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 8. Explore: Phase Diagrams

We sweep two parameters at a time to find which regimes produce star-like vs disordered patterns.

**Sweep A:** k_radial (radial spring strength) vs omega (chirality). We expect:
- High k_radial, low omega: best star_likeness
- High omega: swirl increases, arms rotate
- Low k_radial: agents escape the ring; fragmentation rises

This runs a 5x5 grid of simulations. Each point uses a small grid (N=32) and short runs for speed.

In [ ]:
from chirality.star_ascidian.phase_diagram import run_sweep_A

print("Running 5x5 phase sweep (k_radial vs omega)...")
print("This may take 2-5 minutes on a laptop.")

x_vals, y_vals, grids = run_sweep_A(n_x=5, n_y=5, seed=SEED, verbose=True)

sl_grid = grids["star_likeness"]
swirl_grid = grids["swirl"]

print(f"Sweep done. k_radial range: {x_vals[0]:.1f} to {x_vals[-1]:.1f}")
print(f"            omega range:    {y_vals[0]:.1f} to {y_vals[-1]:.1f}")
print(f"Max star_likeness in sweep: {sl_grid.max():.3f}")
print(f"Max swirl in sweep:         {swirl_grid.max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor=BG)
for ax in axes:
    ax.set_facecolor("#FFFFFF")

# Star-likeness heatmap
im0 = axes[0].imshow(
    sl_grid.T, origin="lower", aspect="auto", cmap="RdYlGn",
    extent=[x_vals[0], x_vals[-1], y_vals[0], y_vals[-1]],
    vmin=0, vmax=1,
)
axes[0].set_xlabel("k_radial (radial spring)", color=INK)
axes[0].set_ylabel("omega (chirality)", color=INK)
axes[0].set_title("Star-likeness score", color=INK)
plt.colorbar(im0, ax=axes[0])

# Swirl heatmap
im1 = axes[1].imshow(
    swirl_grid.T, origin="lower", aspect="auto", cmap="RdBu_r",
    extent=[x_vals[0], x_vals[-1], y_vals[0], y_vals[-1]],
)
axes[1].set_xlabel("k_radial (radial spring)", color=INK)
axes[1].set_ylabel("omega (chirality)", color=INK)
axes[1].set_title("Swirl score (chirality signature)", color=INK)
plt.colorbar(im1, ax=axes[1])

fig.suptitle("Phase diagram: radial spring strength vs chirality", color=INK, fontsize=12)
plt.tight_layout()
plt.show()

best_i, best_j = np.unravel_index(sl_grid.argmax(), sl_grid.shape)
print(f"Best star-likeness at k_radial={x_vals[best_i]:.2f}, omega={y_vals[best_j]:.2f}")

### 8b. Chirality comparison: clean vs twisted

Running the `chiral_twisted_stars` preset lets us see directly how omega changes arm geometry.

In [ ]:
print("Running chiral_twisted_stars preset (omega=2.5)...")
result_chiral = simulate_star_ascidian_colony(
    preset="chiral_twisted_stars",
    seed=SEED,
    n_snapshots=10,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), facecolor=BG)
for ax in axes:
    ax.set_facecolor("#FFFFFF")

def _plot_agents(ax, result, title):
    pos = result.zooid.positions[-1]
    asgn = result.zooid.assignments
    K = result.zooid.K
    L = result.params["L"]
    cm = plt.cm.get_cmap("tab10", K)
    for k in range(K):
        mask = asgn == k
        ax.scatter(pos[mask, 0], pos[mask, 1], c=[cm(k)], s=14, alpha=0.75)
    cx = result.centers[:, 0]
    cy = result.centers[:, 1]
    ax.scatter(cx, cy, c="white", s=60, marker="*", zorder=5,
               edgecolors=INK, linewidths=0.8)
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect("equal")
    ax.set_title(title, color=INK)

_plot_agents(axes[0], result_clean, f"omega=0: clean stars (swirl={result_clean.metrics['swirl_score']:.2f})")
_plot_agents(axes[1], result_chiral, f"omega=2.5: twisted arms (swirl={result_chiral.metrics['swirl_score']:.2f})")

fig.suptitle("Effect of chirality on arm geometry", color=INK, fontsize=12)
plt.tight_layout()
plt.show()

print(f"Clean:  star_likeness={result_clean.metrics['star_likeness_score']:.3f}, swirl={result_clean.metrics['swirl_score']:.3f}")
print(f"Chiral: star_likeness={result_chiral.metrics['star_likeness_score']:.3f}, swirl={result_chiral.metrics['swirl_score']:.3f}")

---
## 9. Compare: Target Features vs Simulation

Now we check each feature from the biological target checklist against what the simulation produces.

In [ ]:
m_clean = result_clean.metrics
m_chiral = result_chiral.metrics

features = [
    ("Multiple star centers",
     f"K={result_clean.zooid.K}",
     "K >= 2",
     result_clean.zooid.K >= 2),
    ("Radial ring structure",
     f"radial_order={m_clean['radial_order']:.2f}",
     ">= 0.5",
     m_clean['radial_order'] >= 0.5),
    ("Arm count near 7",
     f"arm_count={m_clean['arm_count_mean']:.1f}",
     "7 +/- 2",
     abs(m_clean['arm_count_mean'] - 7) <= 2),
    ("Even arm spacing",
     f"uniformity={m_clean['angular_uniformity']:.2f}",
     ">= 0.5",
     m_clean['angular_uniformity'] >= 0.5),
    ("Zooids confined to star",
     f"fragmentation={m_clean['fragmentation']:.2f}",
     "<= 0.2",
     m_clean['fragmentation'] <= 0.2),
    ("Stars not merging",
     f"merge={m_clean['merge_score']:.2f}",
     "<= 0.1",
     m_clean['merge_score'] <= 0.1),
    ("Chirality causes swirl",
     f"swirl_chiral={m_chiral['swirl_score']:.2f}",
     "!= swirl_clean",
     abs(m_chiral['swirl_score'] - m_clean['swirl_score']) > 0.05),
]

print(f"{'Feature':<35} {'Simulation':<30} {'Target':<15} {'Pass?'}")
print("-" * 85)
n_pass = 0
for name, sim_val, target_val, passed in features:
    status = "PASS" if passed else "FAIL"
    if passed:
        n_pass += 1
    print(f"{name:<35} {sim_val:<30} {target_val:<15} {status}")

print()
print(f"Score: {n_pass}/{len(features)} biological features reproduced.")

---
## 10. Explain: What the Model Explains and What It Does Not

### What the model explains

- **Spatial periodicity of star centers**: The GM reaction-diffusion layer produces
  quasi-regular spot spacing controlled by Dh/Da. This matches the characteristic
  inter-star spacing in Botryllus colonies.

- **Radial arm geometry**: The combination of center attraction and a radial spring
  forces zooids to a ring at r_target. Angular repulsion distributes them into discrete arms.

- **Star non-merging**: Sufficiently high Dh/Da gives well-spaced centers.
  Combined with center attraction, zooids stay near their home star.

- **Chirality-induced arm rotation**: A nonzero omega measurably rotates arm
  geometry, producing a nonzero swirl score. Racemic mode (half CW, half CCW)
  gives near-zero net swirl -- a prediction for mixed-chirality colonies.

### What the model does NOT explain

- **Molecular mechanism**: We have no model of the actual signaling molecules
  (Wnt, BMP, Notch) that govern Botryllus patterning. This is a phenomenological
  geometric model, not a mechanistic developmental model.

- **Blastogenic timing**: Botryllus colonies cycle through synchronous blastogenesis
  (budding and regression). Our model has no time-dependent phase structure.

- **Colonial immune recognition**: Botryllus fusibility (the decision to merge or
  reject neighboring colonies) depends on highly polymorphic immune loci.
  Our merge_score only measures geometric proximity, not biology.

- **3D structure**: Real Botryllus colonies are thin sheets with a tunic matrix.
  Our 2D model cannot capture substrate curvature or depth-dependent signaling.

- **Arm count = 7**: The angular repulsion ensures even spacing, but the number
  of arms is set by n_arms (a model parameter), not emergent from the physics.
  The biology that sets 7 arms is not captured.

---
## 11. LLM Workflow: Prompt - Run - Test - Modify - Critique

This section documents how Claude was used in the development of this project.
It follows the hackathon's expected workflow: not vibe-coding blindly, but
using LLM-generated code as a starting point and verifying every output.

### Two best prompts

**Prompt 1: Angular repulsion force design**

```
Write a vectorized numpy function that computes pairwise angular repulsion forces
between active agents. Each agent belongs to an arm group (arm_assignments array).
Force is zero for same-arm pairs. For different-arm pairs at the same center:
  dphi = phi_i - phi_j  (wrapped to [-pi, pi])
  F_angular = -k * (1 - |dphi|/sigma) * sign(dphi) * tangential_hat
Only apply between agents at the same center (assignments array).
Return (N, 2) force array.
```

**Why it worked:** The prompt specified the exact formula and the two assignment arrays
needed to correctly identify same-arm vs. cross-arm pairs. The LLM generated
correct vectorized code on the first try. We verified it by checking that
same-arm force sum was zero and that force direction opposed angular proximity.

**Prompt 2: IMEX scheme for Gierer-Meinhardt**

```
Implement the Gierer-Meinhardt reaction-diffusion system in Fourier space using
an IMEX (implicit-explicit) scheme. Treat diffusion implicitly:
  denom_a = 1 + dt * (Da * k2 + mu_a)
  denom_h = 1 + dt * (Dh * k2 + mu_h)
Treat the nonlinear reaction terms explicitly. Update each step:
  a_hat = (a_hat + dt * reaction_a_hat) / denom_a
  h_hat = (h_hat + dt * reaction_h_hat) / denom_h
The scheme must be unconditionally stable for the diffusion term.
Return activator and inhibitor field histories.
```

**Why it worked:** Specifying the exact denominator structure made the scheme unambiguous.
We verified stability by checking that a large dt (dt=5.0) did not blow up the field.

### Failure case

**The swirl metric broadcast error:**

The LLM-generated `swirl_score` function contained:

```python
# WRONG: produces (N, N) shape due to broadcasting
t_hats = np.column_stack([-r_hats[:, 0][:, None] * 0 - r_hats[:, 1], r_hats[:, 0]])
```

The `[:, None]` created a (N, 1) column that broadcast with the (N,) row
to produce an (N, N) array, not (N, 2). This crashed the dot product.

**Human judgment corrected it:**
```python
# CORRECT: same formula as zooid_agents.py
t_hats = np.column_stack([-r_hats[:, 1], r_hats[:, 0]])
```

The fix was found by comparing the formula against the manually-written zooid_agents.py
and reading the numpy documentation on column_stack broadcasting rules.

### What surprised us

The LLM was better at IMEX schemes and vectorized force computations than expected.
It was worse at getting the broadcast rules right when the formula involved mixing
`[:, None]` with `[:, 0]` in the same expression.

### How we verified outputs

Every new function was verified against at least three checks:
1. Output shape matches expected (e.g., forces are (N, 2))
2. Output is numerically finite (np.all(np.isfinite(...)))
3. Output passes a physical sanity check (e.g., same-arm force sums to zero)

The smoke test (`scripts/06_final_smoke_test.py`) encodes 53 such checks and was
run after every major change.

In [ ]:
# Demonstrate the swirl metric across presets
presets_to_compare = ["clean_star_systems", "chiral_twisted_stars", "noisy_fragmented_systems"]
swirl_values = []
sl_values = []

for preset in presets_to_compare:
    r = simulate_star_ascidian_colony(preset=preset, seed=SEED, n_snapshots=4)
    swirl_values.append(r.metrics["swirl_score"])
    sl_values.append(r.metrics["star_likeness_score"])
    print(f"{preset:<35} swirl={r.metrics['swirl_score']:+.3f}  star_likeness={r.metrics['star_likeness_score']:.3f}")

print()
print("Swirl increases with omega, as expected.")
print("Fragmented systems have lower star_likeness, as expected.")

---
## 12. Five-Slide Checklist

This section serves as a self-check before the presentation.

| Slide | Content | Status |
|---|---|---|
| 1. Target | Botryllus schematic + best simulation still + 1-sentence question | See Section 2 + Section 6 |
| 2. Model | GM equations + zooid force equations + schematic | See Section 4 |
| 3. Simulation | Agent snapshot + radial_order + arm_count + center_spacing | See Section 7 |
| 4. Phase diagram | star_likeness heatmap with regime labels | See Section 8 |
| 5. Insight + LLM | What model explains/does not + best prompts + takeaway | See Section 10-11 |

**Takeaway sentence (for the close of Slide 5):**

> A two-stage mechanism -- a Turing field for center placement plus angular repulsion for arm formation -- can turn local rules into star-shaped colonial order. Chirality adds a global rotation without destroying that order.

### Assets to use in the deck

Run `python scripts/05_make_all_assets.py` to generate all figures. Key files:

| Slide | File |
|---|---|
| 1 | `outputs/panels/slide1_target_and_simulation.png` |
| 2 | `outputs/panels/slide2_model_schematic.png` |
| 3 | `outputs/movies/star_formation_clean.gif` |
| 4 | `outputs/panels/slide4_phase_diagram.png` |
| 5 | `outputs/panels/slide5_insight_and_limits.png` |
| Poster | `outputs/panels/final_summary_panel.png` |